# Example Notebook 06:

Evaluation of Experiment Results

In [ ]:
%load_ext autoreload
%autoreload 2


import config
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient
from statsmodels.stats.anova import anova_lm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from src.evaluation import plot_training_history, prepare_results, plot_metric_boxplot, plot_metrics_scatter_plot, plot_correlation_with_regret, agg_results, display_results

In [ ]:
# Set tracking URI directly to local MLFlow database.
# mlflow.set_tracking_uri("sqlite:///../mlflow.db")
# ... or, set tracking URI to default local port when using MLFlow UI. Run `mlflow ui` in terminal to initialize MLFlow UI.
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow_client = MlflowClient()

## Training of Composite Loss Functions

In [ ]:
# Training History: Cov-e Composite Loss - Direct Training - Germany
ax = plot_training_history(
    mlflow_client,
    data_path=config.DATA_DE_CONFIG['data_path'],
    penalty=config.COV_E_PENALTY_CONFIG['penalty'],
)

plt.show()

In [ ]:
# Training History: Corr-f Composite Loss - Direct Training - Germany
ax = plot_training_history(
    mlflow_client,
    data_path=config.DATA_DE_CONFIG['data_path'],
    penalty=config.CORR_F_PENALTY_CONFIG['penalty'],
)

plt.show()

## Experiment Results

In [ ]:
# prepare results
runs = prepare_results(config.EXP_CONFIG['experiment_name'])

In [ ]:
# Experiment Runs Table
experiment_runs = runs.copy()
metrics_columns = ['regret', 'train_duration_per_epoch', 'mae', 'rmse', 'cov_e', 'corr_f', ]
experiment_runs[metrics_columns+['total_train_duration']] = experiment_runs[metrics_columns+['total_train_duration']].apply(lambda row: display_results(row), axis=1)
experiment_runs

## Training Time

In [ ]:
# ANOVA Test: Training Time ~ Dataset
_runs = runs.copy()
_runs[['training_strategy', 'loss_configuration']] = _runs[['training_strategy', 'loss_configuration']].astype(str)
lm = ols('train_duration_per_epoch ~ C(dataset)', data = _runs).fit()
anova_lm(lm, typ=2).round(3)

In [ ]:
# Results for Training Time
results_by_config = agg_results(runs, ['training_strategy', 'loss_configuration'])
results_by_config['train_duration_relative'] = (results_by_config['train_duration_per_epoch'] / results_by_config[(results_by_config['training_strategy']=='train') & (results_by_config['loss_configuration']=='MAE') ]['train_duration_per_epoch'].min()).round(2)
results_by_config[['training_strategy', 'loss_configuration', 'train_duration_per_epoch', 'train_duration_relative']]

In [ ]:
# Distribution of Training Time by Loss Configuration and Training Strategy
ax = plot_metric_boxplot(runs, 'train_duration_per_epoch')
plt.show()

In [ ]:
# ANOVA Test: Training Time ~ (Training Strategy and Loss Configuration)
_runs = runs.copy()
_runs[['training_strategy', 'loss_configuration']] = _runs[['training_strategy', 'loss_configuration']].astype(str)
lm = ols('train_duration_per_epoch ~ C(training_strategy + loss_configuration)', data = _runs).fit()
anova_lm(lm, typ=2).round(3)

In [ ]:
# Pairwise Tukey HSD Test: Training Time ∼ (Training Strategy + Loss Configuration)
turkey_hsd_duration = pairwise_tukeyhsd(runs['train_duration_per_epoch'], runs['training_strategy'].astype(str) + runs['loss_configuration'].astype(str))
turkey_hsd_duration.summary_frame().round(3)

## Regret


In [ ]:
# ANOVA Test: Regert ~ (Dataset)
_runs = runs.copy()
_runs[['training_strategy', 'loss_configuration']] = _runs[['training_strategy', 'loss_configuration']].astype(str)
lm = ols('regret ~ C(dataset)', data = _runs).fit()
anova_lm(lm, typ=2).round(3)

In [ ]:
# Results for Training Time
results_by_ds_config = agg_results(runs, ['dataset', 'training_strategy', 'loss_configuration'])

### Regret - German Electricity Market

In [ ]:
dataset = config.DATA_DE_CONFIG['data_path']
runs_de = _runs = runs[runs['dataset']==dataset].copy()
results_de = results_by_ds_config[results_by_ds_config['dataset']==dataset].copy()

# Evaluation - German Electricity Market
results_de

In [ ]:
# Regret v. Training Time - German Electricity Market
ax1 = plot_metrics_scatter_plot(
    runs_de,
    results_de
)
ax2 = plot_metric_boxplot(runs_de, 'regret')
plt.show()

In [ ]:
# ANOVA Test: Regret ~ (Training Strategy and Loss Configuration) - German Electricity Market
runs_de[['training_strategy', 'loss_configuration']] = runs_de[['training_strategy', 'loss_configuration']].astype(str)
lm = ols('regret ~ C(training_strategy + loss_configuration)', data = runs_de).fit()
anova_lm(lm, typ=2).round(3)

In [ ]:
# Pairwise Tukey HSD Test: Regret∼(Training Strategy + Loss Configuration) - German Electricity Market
tukey_results = pairwise_tukeyhsd(runs_de['regret'], runs_de['training_strategy'] + runs_de['loss_configuration'])
tukey_results.summary_frame().round(3)

### Regret - Danish Electricity Market

In [ ]:
dataset = config.DATA_DK_CONFIG['data_path']
runs_dk = _runs = runs[runs['dataset'] == dataset].copy()
results_dk = results_by_ds_config[results_by_ds_config['dataset'] == dataset].copy()

# Evaluation - Danish Electricity Market
results_dk

In [ ]:
# Regret v. Training Time - Danish Electricity Market
ax1 = plot_metrics_scatter_plot(
    runs_dk,
    results_dk
)
ax2 = plot_metric_boxplot(runs_dk, 'regret')
plt.show()

In [ ]:
# ANOVA Test: Regret ~ (Training Strategy and Loss Configuration) - Danish Electricity Market
runs_dk[['training_strategy', 'loss_configuration']] = runs_dk[['training_strategy', 'loss_configuration']].astype(str)
lm = ols('regret ~ C(training_strategy + loss_configuration)', data=runs_dk).fit()
anova_lm(lm, typ=2).round(3)

In [ ]:
# Pairwise Tukey HSD Test: Regret∼(Training Strategy + Loss Configuration) - Danish Electricity Market
tukey_results = pairwise_tukeyhsd(runs_dk['regret'], runs_dk['training_strategy'] + runs_dk['loss_configuration'])
tukey_results.summary_frame().round(3)

### Regret - Spanish Electricity Market


In [ ]:
dataset = config.DATA_ES_CONFIG['data_path']
runs_es = _runs = runs[runs['dataset'] == dataset].copy()
results_es = results_by_ds_config[results_by_ds_config['dataset'] == dataset].copy()

# Evaluation - Danish Electricity Market
results_es

In [ ]:
# Regret v. Training Time - Danish Electricity Market
ax1 = plot_metrics_scatter_plot(
    runs_es,
    results_es
)
ax2 = plot_metric_boxplot(runs_es, 'regret')
plt.show()

In [ ]:
# ANOVA Test: Regret ~ (Training Strategy and Loss Configuration) - Danish Electricity Market
runs_es[['training_strategy', 'loss_configuration']] = runs_es[['training_strategy', 'loss_configuration']].astype(str)
lm = ols('regret ~ C(training_strategy + loss_configuration)', data=runs_es).fit()
anova_lm(lm, typ=2).round(3)

In [ ]:
# Pairwise Tukey HSD Test: Regret∼(Training Strategy + Loss Configuration) - Danish Electricity Market
tukey_results = pairwise_tukeyhsd(runs_es['regret'], runs_es['training_strategy'] + runs_es['loss_configuration'])
tukey_results.summary_frame().round(3)

# Correlation of Metrics with Regret



In [ ]:
runs.groupby(['dataset'])[['mae', 'rmse', 'cov_e', 'corr_f']].corrwith(runs['regret']).round(3)

In [ ]:
ax = plot_correlation_with_regret(runs)
plt.show()